In [6]:
import os
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# [중요] OpenAI(챗GPT)를 사용하기 위한 비밀번호(API 키)를 입력하는 곳이에요.
os.environ["OPENAI_API_KEY"] = "여기에_당신의_오픈AI_키를_넣으세요"

def run_rag_chatbot():
    print("🤖: 기숙사 안내 가이드북을 읽고 있습니다. 잠시만 기다려주세요...")

    # 1단계: 파일 가져오기 (문서 로더)
    # 엑셀에서 변환된 CSV 파일을 랭체인이 읽을 수 있도록 가져옵니다.
    loader = CSVLoader(file_path="dormitory_guide_v2.xlsx - 기숙사_운영_데이터.csv", encoding="utf-8")
    docs = loader.load()

    # 2단계: 텍스트 쪼개기 (텍스트 분할기)
    # 컴퓨터가 책을 한 번에 다 읽으면 체하므로, 문장들을 적당한 크기로 쪼갭니다.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    split_docs = text_splitter.split_documents(docs)

    # 3단계: 컴퓨터용 저장소에 저장하기 (임베딩 및 벡터 저장소)
    # 글자를 컴퓨터가 이해하는 숫자로 바꾸어(임베딩) 가상 보관함(Chroma)에 넣습니다.
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)
    
    # 4단계: 검색기 만들기 (리트리버)
    # 사용자가 질문했을 때 보관함에서 가장 알맞은 문서를 찾아오는 '사서' 역할을 합니다.
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

    # 5단계: 챗봇에게 말투와 역할 지정하기 (프롬프트 세팅)
    # 이전 대화처럼 '전문적이고 다정하지만, 규정에는 단호한' 톤앤매너를 지시합니다.
    system_instruction = """
    당신은 기숙사 관리자이자 친절한 인공지능 비서입니다. 
    반드시 주어진 [맥락]의 내용에만 기반하여 사용자의 질문에 답변해 주세요.
    
    [말투 규칙]
    1. 기본 베이스는 늘 예의 바르고 전문적이며 다정한 말투를 사용해 주세요.
    2. 단, '규정'이나 '규칙'에 대한 문의가 있을 때는 다정함을 유지하되, 안 되는 부분은 안 된다고 확실하고 단호하게 답변해 주세요.

    [맥락]
    {context}
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        ("human", "{input}"),
    ])

    # 6단계: 인공지능 두뇌와 체인 연결하기
    # 챗GPT 모델(gpt-4o-mini 등)과 위에서 만든 시스템을 하나로 엮어줍니다.
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    print("🤖: 준비 완료! 기숙사에 대해 궁금한 점을 물어보세요.")
    print("(종료하려면 '종료'를 입력하세요.)\n")

    # 7단계: 챗봇과 대화하기
    while True:
        user_input = input("나: ")
        if user_input in ["종료", "나가기"]:
            print("🤖: 대화를 종료합니다. 안전하고 편안한 기숙사 생활 되세요!")
            break
            
        # 챗봇에게 질문을 던지고 답변을 받아옵니다.
        response = rag_chain.invoke({"input": user_input})
        
        # 답변 출력하기
        print(f"🤖: {response['answer']}\n")

# 프로그램 실행
if __name__ == "__main__":
    run_rag_chatbot()

C:\Users\User\AppData\Local\Temp\ipykernel_20600\540061378.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
c:\Users\User\anaconda3\envs\langchain-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain_chroma'